# Lab 03 — Build a RAG Agent using GitHub Models

In this lab students will build a **Retrieval Augmented Generation (RAG) agent**.

Instead of the model guessing answers, it will:
1. Search a knowledge base
2. Retrieve relevant information
3. Generate an answer grounded in data

Architecture:

User Question  
↓  
Vector Search  
↓  
Relevant Documents  
↓  
LLM Answer

## Install dependencies

In [ ]:
!pip install openai python-dotenv scikit-learn

## Connect to GitHub Models

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.inference.ai.azure.com"
)

print("Connected to GitHub Models")

## Create a Small Knowledge Base

In [ ]:
documents = [
"Python is a programming language widely used for AI and data science.",
"Azure is Microsoft's cloud computing platform.",
"Retrieval Augmented Generation allows models to answer questions using external knowledge.",
"Vector databases help store embeddings for semantic search.",
"Agents combine LLM reasoning with tools and memory."
]

## Convert Text into Vectors

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

## Create a Search Function

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_docs(query):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    best_index = scores.argmax()
    return documents[best_index]

## Test Retrieval

In [ ]:
query = "What is RAG?"

context = search_docs(query)

print("Retrieved document:", context)

## Use Retrieved Context in LLM Prompt

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role":"system","content":"Answer using the provided context."},
        {"role":"user","content":f"Context: {context} \n Question: What is Retrieval Augmented Generation?"}
    ]
)

print(response.choices[0].message.content)

# Exercise

1. Add more documents to the knowledge base
2. Return top‑3 results instead of one
3. Build a chatbot that answers using your documents